# 01 — EDA: MTSamples Clinical Notes *(explained edition)*
**Project:** Clinical Medication Extraction | **Phase 1 of the roadmap**

## What EDA is actually for

EDA is not "make charts of the data." It's **buying information that changes decisions you're about to make**. Every question below exists because a later phase depends on the answer:

| Question | Decision it changes |
|---|---|
| Q1 Data health | whether your eval metrics will be honest |
| Q2 Length | whether Phase 5 needs LLM chunking |
| Q3 Specialty mix | how Phase 3 samples the gold set |
| Q4 Section headers | the entire Phase 2 sectionizer design |
| Q5 Medication vocabulary | how the Phase 3 drug lexicon must be built |
| Q6 Reading notes | everything — this is the highest-value cell |

A chart that changes nothing is decoration. If you can't name the decision, skip the chart.

**Everything in this notebook runs and is interpreted for you.** Read the interpretation after each output — that's the part that transfers to your next project.

## Setup

⚠️ **Set `BASE` to your own Drive folder.** Everything else is relative to it.

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import plotly.express as px

pd.set_option('display.max_colwidth', 120)

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    pass

# <<< EDIT THIS LINE if your folder differs >>>
BASE = '/content/drive/MyDrive/Clinical_notes/' if IN_COLAB else ''

RAW  = BASE + 'raw/'
WORK = BASE + 'working/'
FIGS = BASE + 'figures/'

import os
for p in (RAW, WORK, FIGS):
    os.makedirs(p, exist_ok=True)

# Path check BEFORE loading — catches the most common Colab error early
print('BASE =', BASE)
print('raw/ contains:', os.listdir(RAW) if os.path.isdir(RAW) else 'MISSING')

Mounted at /content/drive
BASE = /content/drive/MyDrive/Clinical_notes/
raw/ contains: ['mtsamples.csv']


In [2]:
df = pd.read_csv(RAW + 'mtsamples.csv', index_col=0)
df.columns = [c.strip() for c in df.columns]
print('Shape:', df.shape)
df.head(3)

Shape: (4999, 5)


,description,medical_specialty,sample_name,transcription,keywords
0,A 23-year-old white female presents with complaint of allergies.,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female presents with complaint of allergies. She used to have allergies when s...","allergy / immunology, allergic rhinitis, allergies, asthma, nasal sprays, rhinitis, nasal, erythematous, allegra, sp..."
1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climbing stairs, difficulty with airline seats, tying shoes, used to public...","bariatrics, laparoscopic gastric bypass, weight loss programs, gastric bypass, atkin's diet, weight watcher's, body ..."
2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC today. He is a very pleasant gentleman who is 42 years old, 344 pound...","bariatrics, laparoscopic gastric bypass, heart attacks, body weight, pulmonary embolism, potential complications, sl..."


### What you're looking at

5 columns, ~5,000 rows. The one that matters is `transcription` — the note text. `medical_specialty` and `sample_name` are metadata that will let you stratify samples later. `description` and `keywords` are website scaffolding from mtsamples.com, not clinical content.

**These are real transcribed medical reports** — dictated by clinicians, typed by transcriptionists, published as teaching samples. Not EHR exports. That distinction matters and we'll come back to it at the end.

---
# Q1 — Data health
**Decision it changes:** whether your Phase 3 evaluation metrics mean anything.

In [3]:
print('Nulls per column:')
print(df.isna().sum().to_string())
print()
print('Duplicate transcription texts:', df['transcription'].duplicated().sum())

Nulls per column:
description             0
medical_specialty       0
sample_name             0
transcription          33
keywords             1068

Duplicate transcription texts: 2641


### 🚨 Interpretation — the finding that matters most in this notebook

**33 null transcriptions** — trivial, drop them.

**1,068 null keywords** — irrelevant, we don't use that column.

**2,609 duplicate transcriptions.** Over *half* the dataset. This is not a minor cleanup detail; it's the kind of thing that quietly invalidates a whole project.

Why it happens: mtsamples.com cross-lists the same note under multiple specialties. One EGD procedure note is filed under both *Surgery* and *Gastroenterology*, so it appears twice with identical text.

**Why it would wreck your metrics:** in Phase 3 you'll split notes into a gold/eval set and a development set. If note X is in both — under two different specialty labels — your extractor gets tuned on text it will later be scored on. Your F1 comes back beautifully high and means nothing. This is **data leakage**, and it's the single most common way ML results turn out to be fake.

You've met this exact shape before in Epic Clarity: joining a table at the wrong grain and getting duplicated rows that inflate every count downstream. Same bug, different data. The instinct — *check the grain before you trust the number* — is the transferable part.

**Decision:** deduplicate on `transcription` for all modeling and evaluation. Log it.

In [4]:
# Look at an actual duplicate pair to make it concrete
dups = df[df['transcription'].duplicated(keep=False)].sort_values('transcription')
pair = dups.head(2)
print('Same text, filed under two specialties:')
print(pair[['medical_specialty', 'sample_name']].to_string())
print()
print('First 150 chars of each — identical:')
for t in pair['transcription']:
    print(' ', repr(t[:150]))

Same text, filed under two specialties:
      medical_specialty                    sample_name
874             Surgery   EGD With Photos & Biopsies. 
3577   Gastroenterology   EGD With Photos & Biopsies. 

First 150 chars of each — identical:
  '1.  Odynophagia.,2.  Dysphagia.,3.  Gastroesophageal reflux disease rule out stricture.,POSTOPERATIVE DIAGNOSES:,1.  Antral gastritis.,2.  Hiatal hern'
  '1.  Odynophagia.,2.  Dysphagia.,3.  Gastroesophageal reflux disease rule out stricture.,POSTOPERATIVE DIAGNOSES:,1.  Antral gastritis.,2.  Hiatal hern'


In [5]:
d = (df
     .dropna(subset=['transcription'])
     .drop_duplicates(subset='transcription', keep='first')
     .copy())
d['medical_specialty'] = d['medical_specialty'].str.strip()

print(f'{len(df)} raw rows  ->  {len(d)} unique notes')
print(f'We just discarded {len(df) - len(d)} rows ({100*(len(df)-len(d))/len(df):.0f}%) — '
      'and that is the correct call.')

4999 raw rows  ->  2357 unique notes
We just discarded 2642 rows (53%) — and that is the correct call.


---
# Q2 — How long are these notes?
**Decision it changes:** whether Phase 5 (LLM extraction) needs a chunking strategy.

Chunking — splitting a long note into pieces because it won't fit in a model's context window — adds real complexity: you have to stitch results back together and handle entities that straddle a boundary. **If we don't need it, that's an evening saved.**

In [6]:
d['char_len'] = d['transcription'].str.len()
d['word_len'] = d['transcription'].str.split().str.len()

fig = px.histogram(d, x='word_len', nbins=60,
                   title='Note length (words) — deduplicated corpus',
                   labels={'word_len': 'words per note'})
fig.add_vline(x=d['word_len'].median(), line_dash='dash',
              annotation_text=f"median {int(d['word_len'].median())}")
fig.write_html(FIGS + 'length_hist.html')
fig.show()

print(f"median : {d['word_len'].median():.0f} words")
print(f"90th %  : {d['word_len'].quantile(0.9):.0f} words")
print(f"max     : {d['word_len'].max():.0f} words")

median : 389 words
90th %  : 843 words
max     : 3029 words


### Interpretation

Median ~389 words, 90th percentile ~843, longest ~3,029.

A rough conversion: **1 word ≈ 1.3 tokens** for clinical text (drug names fragment into several subword tokens — you'll see exactly why in Phase 4). So the longest note is roughly 4,000 tokens.

Every instruct model you'd realistically use in Phase 5 (Qwen2.5, Llama 3.1) handles 32k+ tokens of context. **Conclusion: no chunking needed.** Whole notes go in as single prompts.

That's a genuine decision made cheaply — and notice the shape of it: you didn't need a chart to *look nice*, you needed one number compared against one threshold. Write it in `decisions.md` and move on.

⚠️ **Flag for later:** real MIMIC discharge summaries run far longer (copy-forward text, embedded lab tables). Revisit this when credentialing lands. It's a genuine limitation of developing on MTSamples, and naming it in your README is a strength, not a weakness.

In [7]:
top10 = d['medical_specialty'].value_counts().head(10).index
fig = px.box(d[d['medical_specialty'].isin(top10)],
             x='medical_specialty', y='word_len',
             hover_data=['sample_name'],
             title='Note length by specialty — hover any outlier to see which note it is')
fig.update_xaxes(tickangle=35)
fig.write_html(FIGS + 'length_by_specialty.html')
fig.show()

**Why Plotly instead of seaborn here:** the hover tooltip. With text data, "which note is that outlier?" is the question you always want next, and a static chart can't answer it. Interactivity isn't decoration in text EDA — it's how you get from a distribution back to the actual document.

---
# Q3 — What kinds of notes are these?
**Decision it changes:** how Phase 3 stratifies the gold-set sample.

In [8]:
counts = d['medical_specialty'].value_counts()
fig = px.bar(counts.head(15)[::-1], orientation='h',
             title='Unique notes per specialty (top 15)',
             labels={'value': 'notes', 'index': ''})
fig.update_layout(showlegend=False)
fig.write_html(FIGS + 'specialty_counts.html')
fig.show()
print(counts.head(8).to_string())

medical_specialty
Surgery                          976
Radiology                        248
General Medicine                 157
Urology                          156
SOAP / Chart / Progress Notes    144
Neurology                         67
Orthopedic                        56
Consult - History and Phy.        55


### Interpretation

**Surgery dominates — 976 of 2,357 notes (41%).** Operative reports, largely: pre-op diagnosis, procedure description, findings. They're long, formulaic, and **medication-poor** — an op note lists anesthesia agents, not a patient's home medication list.

That's a mismatch with your target task. Extracting medications from a corpus that's mostly operative reports means most of your effort goes into notes that have nothing to extract.

**So we narrow to a working subset**: the note types where medications genuinely live — progress notes, general medicine visits, discharge summaries.

In [9]:
mask = (d['medical_specialty'].isin(['SOAP / Chart / Progress Notes', 'General Medicine'])
        | d['sample_name'].str.contains('Discharge', case=False, na=False))
work = d[mask].copy()

print('Working subset:', len(work), 'notes')
print()
print(work['medical_specialty'].value_counts().head(6).to_string())
print()
print(f"median length: {work['word_len'].median():.0f} words "
      f"(max {work['word_len'].max():.0f})")

Working subset: 373 notes

medical_specialty
General Medicine                 157
SOAP / Chart / Progress Notes    144
Discharge Summary                 13
Orthopedic                        12
Gastroenterology                   8
Pediatrics - Neonatal              8

median length: 388 words (max 1403)


### Interpretation — and an honest constraint

**373 notes.** Two categories carry it: General Medicine (157) and SOAP/Progress Notes (144), with a long tail of discharge-type notes across specialties.

Now the Phase 3 sampling question. You need ~75 notes for a gold set, drawn from 373.

- **Proportional stratification** (sample each category in proportion to its size) gives an eval set that mirrors the corpus. Metrics reflect "how well does this work on data like this?"
- **Equal stratification** (same number per category) gives better precision on per-category performance, but the headline metric no longer reflects the real mix.

**For this project: proportional.** You have one headline number to report and you want it to be honest about the corpus. Your RCT instincts apply exactly here — this is the same tradeoff as sampling for representativeness versus subgroup power, and with 75 units you don't have the budget for subgroup power.

**The real constraint to name out loud:** 373 notes is *small*. Big enough for a credible baseline and eval harness; too small for fine-tuning anything. That's fine — it's precisely why the Phase 4 plan uses pretrained models off the shelf and defers fine-tuning until MIMIC access. The dataset size chose the modeling strategy, not the other way around. Say that in an interview and you sound like someone who has actually run projects.

---
# Q4 — Section headers
**Decision it changes:** the entire design of the Phase 2 sectionizer.

First, the structural discovery — run this before anything else.

In [10]:
one = work['transcription'].iloc[0]
print('Contains newline characters?', '\n' in one)
print('Fraction of notes with ANY newline:',
      work['transcription'].str.contains('\n').mean())
print()
print(repr(one[:600]))

Contains newline characters? False
Fraction of notes with ANY newline: 0.0

'HISTORY OF PRESENT ILLNESS:,  The patient is a 17-year-old female, who presents to the emergency room with foreign body and airway compromise and was taken to the operating room.  She was intubated and fishbone.,PAST MEDICAL HISTORY: , Significant for diabetes, hypertension, asthma, cholecystectomy, and total hysterectomy and cataract.,ALLERGIES:  ,No known drug allergies.,CURRENT MEDICATIONS: , Prevacid, Humulin, Diprivan, Proventil, Unasyn, and Solu-Medrol.,FAMILY HISTORY: , Noncontributory.,SOCIAL HISTORY: , Negative for illicit drugs, alcohol, and tobacco.,PHYSICAL EXAMINATION:  ,Please se'


### 🔑 Interpretation — the finding that shapes Phase 2

**Zero notes contain a newline character.**

Look at the raw string: `'HISTORY OF PRESENT ILLNESS:,  The patient is a 17-year-old female...fishbone.,PAST MEDICAL HISTORY: , Significant for diabetes...'`

The notes were dictated, transcribed, then flattened into single-line CSV cells. **The comma is doing the job a line break would normally do.**

Why this is a big deal: every sectionizer tutorial you will find online assumes newlines — split into lines, check whether each line looks like a header. **That approach is impossible here.** You have to locate headers *inside* one continuous string, which puts all the burden on a carefully anchored regex.

This is the most realistic lesson in the notebook: **real clinical text arrives in whatever shape the source system dumped it in, and step one is always discovering that shape rather than assuming it.** You found this by printing `repr()` of a raw string — the least glamorous, most reliable move in data work.

In [11]:
# Census of header-like strings. Anchored to a delimiter to avoid mid-sentence matches.
header_pat = re.compile(r'(?:^|[,.;:]\s*)([A-Z][A-Z0-9 /&()\'-]{1,60}?)\s*:')

census = Counter()
for text in work['transcription']:
    census.update(h.strip() for h in header_pat.findall(text))

print('Distinct header strings:', len(census))
top30 = pd.DataFrame(census.most_common(30), columns=['header', 'count'])
fig = px.bar(top30[::-1], x='count', y='header', orientation='h',
             title='Top 30 section headers in the working subset', height=750)
fig.write_html(FIGS + 'header_census.html')
fig.show()

Distinct header strings: 486


In [12]:
singletons = [h for h, c in census.items() if c == 1]
print(f'{len(singletons)} of {len(census)} headers appear exactly once')
print()
print('Sample of the long tail:')
print(singletons[:25])

273 of 486 headers appear exactly once

Sample of the long tail:
['PERTINENT LABORATORIES', 'DISCHARGE FOLLOWUP PLANNING', 'ADDITIONAL GOALS', 'NUMBER OF SESSIONS COMPLETED', 'LMP', 'SOCIAL', 'PSYCHE', 'ASSESSMENT & PLAN', 'PR', 'TREATMENT PLAN', 'HISTORY OF THE PRESENT ILLNESS', 'VISCOSUPPLEMENTATION IN PAST', 'VAS PAIN SCORE', 'WOMAC SCORE', 'A-1 WOMAC SCORE', 'CV - RESP', 'PRINCIPAL DIAGNOSES', 'INTERIM HISTORY', 'AXILLA', 'TESTING OF STATION AND GAIT', 'CHART NOTE', 'SOCIOECONOMIC STATUS', 'PAST MEDICAL HISTORY/SURGERIES/HOSPITALIZATIONS', 'PAST MEDICAL HISTORY / SURGERY / HOSPITALIZATIONS', 'FAMILY HISTORY / PERSONAL HISTORY']


### Interpretation — a head and a tail, and they need different treatment

**The head (~50 headers) covers the vast majority of content.** `HISTORY OF PRESENT ILLNESS`, `PHYSICAL EXAMINATION`, `ALLERGIES`, `MEDICATIONS`, `ASSESSMENT`, `PLAN`. These are the ones worth mapping by hand.

**The tail is ~277 one-off strings**, and it's a mix of three different things:
1. **Genuine rare headers** — `PERTINENT LABORATORIES`, `ADVANCED DIRECTIVE`, `VAS PAIN SCORE`. Real, just uncommon.
2. **Spelling variants of common headers** — `ASSESSMENT & PLAN` vs `ASSESSMENT AND PLAN`, `PHYSICAL EXAM` vs `PHYSICAL EXAMINATION`.
3. **Regex artifacts** — strings the pattern matched that aren't headers at all.

**Three design consequences for Phase 2**, each of which you'd otherwise discover the hard way:

- **You need normalization, not just detection.** `PHYSICAL EXAM` and `PHYSICAL EXAMINATION` must collapse to one key, or downstream code has to know every spelling.
- **A hand-built dictionary beats fuzzy matching here.** ~50 real types is a 20-minute dictionary that is exact and debuggable. Fuzzy matching would introduce its own failure mode — and collapsing `DISCHARGE MEDICATIONS` into `MEDICATIONS` is a clinically dangerous merge. **Choose the boring tool when the boring tool is correct.**
- **Never silently drop unknown headers.** Keep them under a cleaned-up key. Content that vanishes without an error is the worst kind of bug — nothing looks wrong.

You'll also notice `HEENT`, `ABDOMEN`, `NECK` ranking high. Those aren't top-level sections — they're **sub-headings inside the physical exam**. Phase 2 namespaces them `exam:*` so medication extraction can skip them wholesale, which removes false-positive surface area for free.

---
# Q5 — Where are the medications, and what do they look like?
**Decision it changes:** how the Phase 3 drug lexicon gets built. This one has a surprise.

In [13]:
seed_drugs = ['amiodarone', 'sotalol', 'metoprolol', 'atenolol', 'digoxin',
              'warfarin', 'aspirin', 'lisinopril', 'furosemide', 'atorvastatin',
              'insulin', 'metformin', 'prednisone', 'albuterol', 'omeprazole',
              'gabapentin', 'oxycodone', 'acetaminophen', 'ibuprofen', 'azithromycin']

lower = work['transcription'].str.lower()
hits_per_note = lower.apply(lambda t: sum(dr in t for dr in seed_drugs))

print(f'Notes containing >=1 seed drug: {(hits_per_note > 0).sum()} / {len(work)} '
      f'({100*(hits_per_note > 0).mean():.0f}%)')
print()
counts = {dr: int(lower.str.contains(dr, regex=False).sum()) for dr in seed_drugs}
print(pd.Series(counts).sort_values(ascending=False).to_string())

Notes containing >=1 seed drug: 158 / 373 (42%)

aspirin          53
insulin          31
prednisone       27
lisinopril       23
albuterol        19
metoprolol       17
digoxin          15
ibuprofen        14
atenolol         13
metformin        12
omeprazole       12
furosemide       10
oxycodone         8
acetaminophen     5
gabapentin        4
amiodarone        4
azithromycin      3
warfarin          3
atorvastatin      1
sotalol           0


### First read: only 42% of notes hit our 20-drug list

Before concluding "these notes are medication-poor," ask the better question: **is the list wrong, or is the corpus wrong?** Let's look at what's actually written in medication sections.

In [14]:
# Pull text right after a MEDICATIONS header and count capitalized words
med_pat = re.compile(r'MEDICATIONS?:\s*,?\s*(.{0,400})')
brand_tokens = Counter()
for t in work['transcription']:
    for m in med_pat.finditer(t):
        brand_tokens.update(re.findall(r'\b[A-Z][a-z]{3,}\b', m.group(1)))

noise = {'None','Blood','Vital','General','Temperature','This','Pulse','Signs','Weight',
         'Status','Significant','Please','There','Mother','Father','Patient','Noncontributory'}
drugs_found = [(w, n) for w, n in brand_tokens.most_common(60) if w not in noise][:25]
print('Capitalized words appearing in medication sections:')
for w, n in drugs_found:
    print(f'  {n:3d}  {w}')

Capitalized words appearing in medication sections:
   15  Lasix
   14  Synthroid
   13  Vicodin
   11  Aspirin
   11  Tylenol
   10  Coumadin
   10  Lipitor
    9  Colace
    9  Advair
    9  Claritin
    8  Toprol
    7  Flomax
    7  Paxil
    7  Albuterol
    7  Lantus
    6  Nexium
    6  Ativan
    6  Coreg
    6  Include
    6  Protonix
    6  Skin
    6  Clear
    6  Lortab
    6  Refer
    6  Neck


### 🚨 Interpretation — the finding that changes Phase 3

Look at that list: **Lasix, Synthroid, Vicodin, Coumadin, Lipitor, Colace, Advair, Claritin, Toprol, Flomax, Paxil, Lantus.**

These are **brand names**. Our seed list was almost entirely generic names — and that's why it only caught 42% of notes. It's not that the corpus lacks medications; it's that our lexicon spoke the wrong dialect.

Notice `Lasix` (15) vastly outnumbers `furosemide` (10), and `Coumadin` (10) beats `warfarin` (3). These are literally the same drugs. **Clinicians dictate brand names**, so any lexicon built only from generic names will silently miss half of everything — and "silently" is the dangerous word. Your recall would be poor, and nothing in your code would error.

**Three consequences for Phase 3:**

1. **The lexicon must include brand names.** Your CredibleMeds QT list is generic-heavy; supplement it with RxNorm, which carries brand↔generic relationships as structured data. This is not optional — it's the difference between ~50% and ~90% recall.
2. **You need brand→generic normalization**, which is exactly what the Phase 5 RAG component does. That component just went from "nice architectural showpiece" to **load-bearing**. Good — RAG that solves a real problem interviews far better than RAG bolted on for buzzword value.
3. **Case matters as a signal.** Brand names are capitalized mid-sentence; generic names usually aren't. That's a cheap feature for the rule-based extractor, though it will misfire at sentence starts — a known tradeoff to log.

**The meta-lesson, and it's the one to keep:** a low number is a question, not an answer. "42% hit rate" could have meant *the corpus is unsuitable* — a project-killing conclusion. One more query showed it meant *our lexicon was wrong* — a two-hour fix. **Always ask whether the disappointing number is telling you about the data or about your instrument.**

---
# Q6 — Read the notes
**Decision it changes:** all of them. No tooling; just read.

In [15]:
for i, (_, row) in enumerate(work.sample(5, random_state=42).iterrows(), 1):
    print('=' * 88)
    print(f"NOTE {i} | {row['medical_specialty'].strip()} | {row['sample_name'].strip()}")
    print('=' * 88)
    print(row['transcription'][:2200])
    print()

NOTE 1 | General Medicine | Consult - Hypertension
HISTORY OF PRESENT ILLNESS:,  The patient is a 74-year-old white woman who has a past medical history of hypertension for 15 years, history of CVA with no residual hemiparesis and uterine cancer with pulmonary metastases, who presented for evaluation of recent worsening of the hypertension.  According to the patient, she had stable blood pressure for the past 12-15 years on 10 mg of lisinopril.  In August of 2007, she was treated with doxorubicin and, as well as Procrit and her blood pressure started to go up to over 200s.  Her lisinopril was increased to 40 mg daily.  She was also given metoprolol and HCTZ two weeks ago, after she visited the emergency room with increased systolic blood pressure.  Denies any physical complaints at the present time.  Denies having any renal problems in the past.,PAST MEDICAL HISTORY:,  As above plus history of anemia treated with Procrit.  No smoking or alcohol use and lives alone.,FAMILY HISTORY:,  Un

### Interpretation — what to notice, and why each thing matters

Reading these five, here's what's visible and what it implies:

**1. Medication phrasing is wildly inconsistent.** You'll see bare lists (`Prevacid, Humulin, Diprivan`), full sigs (`Bactrim DS one tablet p.o. b.i.d. for ten days`), and narrative mentions (`she was started on prednisone`). → Phase 3's dose/route/frequency regexes need to handle *absence* gracefully. A drug with no dose is still a valid extraction, not a failure.

**2. Abbreviations are Latin and everywhere.** `p.o.`, `b.i.d.`, `q.d.`, `p.r.n.`, `q.h.s.` — with inconsistent periods (`bid` vs `b.i.d.`). → Normalize these to a canonical frequency vocabulary. Note the trap: the periods interact with the comma-as-delimiter structure, so a naive sentence splitter will shatter `b.i.d.` mid-token.

**3. Negation and history are pervasive.** `No known drug allergies`, `denies tobacco`, `was previously on`, `discontinued`. → Every one of these is a false positive waiting to happen. A drug mentioned in the note is *not* the same as a drug the patient is taking. This is why section context (Q4) plus negation handling (Phase 2) exist.

**4. Placeholder de-identification.** `Dr. X`, `Mr. ABC`, `room 123`. Real de-identified corpora use surrogates instead — which is why `DR` needed to be a stopword in the header filter (`Dr. X:` matches a header pattern perfectly).

**5. Content pointers instead of content.** `Please see the hospital chart.` appears as an entire section body. → Some sections are structurally present but semantically empty. Your coverage metric will count them as covered; a human wouldn't. Worth knowing your metric's blind spot.

**The habit:** none of this came from a chart. Reading 5 notes for 15 minutes produced more design constraints than every visualization above. Do this on every text project, first, always.

---
# Save and wrap up

In [16]:
work_out = work.drop(columns=['char_len', 'word_len'])
work_out.to_parquet(WORK + 'notes_subset.parquet')
print('Saved:', WORK + 'notes_subset.parquet', '| notes:', len(work_out))
print('Figures saved to:', FIGS)

Saved: /content/drive/MyDrive/Clinical_notes/working/notes_subset.parquet | notes: 373
Figures saved to: /content/drive/MyDrive/Clinical_notes/figures/


## What this EDA bought you

Six decisions, each made from evidence rather than assumption:

| # | Finding | Decision |
|---|---|---|
| 1 | 2,609 duplicate texts (cross-listed specialties) | deduplicate before any split — avoids leakage |
| 2 | median 389 words, max ~3,000 | no LLM chunking needed; revisit for MIMIC |
| 3 | Surgery dominates; working subset = 373 notes | proportional stratification; too small to fine-tune |
| 4 | zero newlines; ~50 core headers + long tail | regex sectionizer with hand-built normalization map |
| 5 | brand names dominate (Lasix > furosemide) | lexicon needs RxNorm brand names; RAG normalization is load-bearing |
| 6 | negation, Latin abbreviations, empty sections | negation handling + frequency normalization required |

**Copy into `decisions.md`** — all six, one line each. That file is your design doc accumulating for free.

### The honest limitation to carry into your README

These are **transcription teaching samples**, not EHR exports. Compared to real MIMIC notes they lack copy-forward bloat, EHR templating artifacts, embedded lab tables, and true de-identification surrogates. Everything you build here will need re-validation on MIMIC.

That's not a weakness in your project — **it's a planned generalization test.** "Developed on MTSamples, validated on MIMIC-IV-Note, with a documented transfer-gap analysis" is a stronger story than starting on MIMIC would have been.

**Next: `02_sectionizer.ipynb`** — turn the Q4 header census into `split_sections()`.